In [17]:
import torch
torch.set_default_dtype(torch.float64)
from cpu_cb import E8P12_codebook

In [18]:
def hb_transform(x):
    if len(x.shape) == 1:
        x = x.view(1,-1)
    (m,n) = x.shape
    k = 1
    while 4**k < n:
        k += 1
    assert(4**k == n)
    b = torch.tensor([1,1,-1,-1]).to(x.dtype)
    x = x.reshape((m,) + (4,)*k)
    for i in range(k):
        x = x.flip(1+i) + x * b.view((1,)*(i+1) + (4,) + (1,)*(k-1-i))
    x = x.reshape((m,) + (2,2)*k)
    x = x.permute((0,) + tuple(2*i+1 for i in range(k)) + tuple(2*i+2 for i in range(k)))
    return x.reshape(m, 2**k, 2**k) / (2**k)

In [19]:
def calculate_k(n):
    k = 1
    while 4**k < n:
        k += 1
    assert(4**k == n)
    return k

In [20]:
def hb_transform_loop(x, k): # hb_transform without the end permutate and reshape
    (m,n) = x.shape
    assert 4**k == n
    b = torch.tensor([1, 1, -1, -1]).to(x.dtype)
    x = x.reshape((m,) + (4,) * k)
    for i in range(k):
        x = x.flip(1 + i) + x * b.view((1,) * (i + 1) + (4,) + (1,) * (k - 1 - i))
    # return x.reshape((m,) + (2, 2) * k) / (2**k)
    return x.reshape((m,) + (2, 2) * k)

In [21]:
def hb_transform_reshape(x, k): # given original hb_transform function shape go to the hb_transform_loop
    m = x.shape[0]
    x = x.reshape((m,) + (2,) * (2 * k))
    forward_perm = [0] + [2 * i + 1 for i in range(k)] + [2 * i + 2 for i in range(k)]
    inverse_perm = [0] * (2 * k + 1)
    for i, p in enumerate(forward_perm):
        inverse_perm[p] = i 
    x = x.permute(inverse_perm)
    return x.reshape((m,) + (4,) * k)

In [22]:
cliques = [
    [0, 2, 11, 25, 33, 39, 47, 57],
    [1, 3, 10, 24, 32, 38, 46, 56],
    [4, 6, 15, 29, 35, 37, 43, 61],
    [5, 7, 14, 28, 34, 36, 42, 60],
    [12, 18, 20, 26, 44, 53, 55, 62],
    [13, 19, 21, 27, 45, 52, 54, 63],
    [8, 16, 22, 30, 40, 49, 51, 58],
    [9, 17, 23, 31, 41, 48, 50, 59]
]
cb = E8P12_codebook()

In [23]:
def bt(M, n):
    k = calculate_k(n * n)
    M_flat = hb_transform_reshape(M.reshape(1, n, n), k).reshape(1, n * n)
    result = hb_transform_loop(M_flat, k) / (2**k)
    return result.reshape(n * n) / (1.0 / n)

def ibt(error, n):
    return hb_transform(error.reshape(1, n * n)).reshape(n, n)

In [24]:
B_2 = hb_transform(torch.eye(4)) * 2
basis_mats = hb_transform(torch.eye(64)) * 8
def fast_orth_quant(coeffs, H_sqrt, n, hat_coeffs, error):
    tr_H_sqrt = torch.diagonal(H_sqrt).sum()
    if n == 8:
        for c in range(8):
            target = coeffs[cliques[c]].clone()
            for cp in range(c):
                B_curr = basis_mats[cliques[c]]
                B_prior = basis_mats[cliques[cp]]
                cross = ((B_curr @ H_sqrt).unsqueeze(1) * B_prior.unsqueeze(0)).sum(dim=(-2, -1)) / tr_H_sqrt / 64
                target = target + cross @ error[cliques[cp]]
            hat_coeffs[cliques[c]] = cb.quantize(target.unsqueeze(0))[0].squeeze(0)
            error[cliques[c]] = coeffs[cliques[c]] - hat_coeffs[cliques[c]]
        return
    next_n = n // 2
    Ha, Hb = H_sqrt[:next_n, :next_n], H_sqrt[:next_n, next_n:]
    Hc, Hd = H_sqrt[next_n:, :next_n], H_sqrt[next_n:, next_n:]
    H_decomp = [(Ha+Hd)/2, (Hb+Hc)/2, (Hb-Hc)/2, (Ha-Hd)/2]
    cliques_per_group = (n * n) // 32
    group_size = (n * n) // 4
    clique_vals = torch.stack([torch.tensor(cliques[c % 8], dtype=torch.long) + 64 * (c // 8) for c in range(cliques_per_group)])
    for b in range(4):
        corrections = torch.zeros(cliques_per_group, 8, dtype=coeffs.dtype)
        for a in range(b):
            d, sign = 4, 0
            for i in range(4):
                if torch.allclose(B_2[i] @ B_2[a], B_2[b]):
                    d, sign = i, +1
                    break
                if torch.allclose(B_2[i] @ B_2[a], -B_2[b]):
                    d, sign = i, -1
                    break
            e_a = error[a * group_size:(a+1) * group_size]
            f = bt(ibt(e_a, next_n) @ H_decomp[d], next_n)
            corrections += sign * f[clique_vals] / tr_H_sqrt / next_n**2
        coeffs_b = coeffs[b * group_size:(b+1) * group_size].clone()
        for c_ind in range(cliques_per_group):
            coeffs_b[clique_vals[c_ind]] += corrections[c_ind]
        hat_b = hat_coeffs[b * group_size:(b+1) * group_size]
        error_b = error[b * group_size:(b+1) * group_size]
        fast_orth_quant(coeffs_b, H_decomp[0], next_n, hat_b, error_b)

def fast_orth_quant_full(Wr, Hr, n_full, n_sub=64):
    """
    Outer LDLQ loop:  inter-group error propagation via full Cholesky of Hr
    Inner fast_orth_quant: intra-group recursive J corrections (unchanged)
    Total complexity: O(n³) — Cholesky dominates
    """
    dev, dtype = Wr.device, Wr.dtype
    L    = torch.linalg.cholesky(Hr)          # O(n_full³), done once
    hatWr = torch.zeros_like(Wr)

    for col in reversed(range(n_full // n_sub)):
        c0, c1 = col * n_sub, (col + 1) * n_sub

        # ── Outer LDLQ correction: error from ALL right-side column groups ──
        if c1 < n_full:
            outer_corr = (Wr[:, c1:] - hatWr[:, c1:]) @ L[c1:, c0:c1]
        else:
            outer_corr = torch.zeros(n_full, n_sub, dtype=dtype, device=dev)

        W_group = Wr[:, c0:c1] + outer_corr      # (n_full, n_sub)

        # Local H for this column group (8×8 blocks only need this sub-H)
        H_sub      = Hr[c0:c1, c0:c1]
        S_s, V_s   = torch.linalg.eigh(H_sub)
        H_sqrt_sub = V_s @ torch.diag(S_s.clamp(min=0).sqrt()) @ V_s.T

        # ── Inner fast_orth_quant: rows are independent, process each row group ──
        k_s = calculate_k(n_sub * n_sub)
        for row in range(n_full // n_sub):
            r0, r1  = row * n_sub, (row + 1) * n_sub
            W_block = W_group[r0:r1, :]           # (n_sub, n_sub)

            # Encode to coefficient space
            W_flat  = W_block.reshape(1, 2**k_s, 2**k_s)
            coeffs  = (hb_transform_loop(
                           hb_transform_reshape(W_flat, k_s).reshape(1, n_sub*n_sub),
                           k_s) / (2**k_s)).reshape(n_sub*n_sub) * n_sub
            Wscale  = coeffs.square().mean().sqrt() / cb.opt_scale
            coeffs  = coeffs / Wscale

            hat = torch.zeros_like(coeffs)
            err = torch.zeros_like(coeffs)

            # Intra-group recursive J correction — structure unchanged
            fast_orth_quant(coeffs, H_sqrt_sub, n_sub, hat, err)

            hat = hat * Wscale
            hatWr[r0:r1, c0:c1] = hb_transform(
                hat.reshape(1, n_sub*n_sub)).reshape(n_sub, n_sub)

    return hatWr

In [25]:
n = 64
k = calculate_k(n * n)
torch.manual_seed(0)
W = torch.randn(n, n)
A = torch.randn(n, n)
H = (A + A.T) / 2
H = H + torch.eye(n) * (abs(torch.linalg.eigvalsh(H).min()) + 1.0)
hatW_fast = fast_orth_quant_full(W, H, n_full=n, n_sub=64)
err = (W - hatW_fast).norm() / W.norm()
print(f"Error: {err.item():.6f}")
proxy_loss = ((hatW_fast - W) @ H @ (hatW_fast - W).T).trace()
print(f"Proxy loss: {proxy_loss.item():.6e}")

Error: 0.300115
Proxy loss: 4.297903e+03


In [26]:
# old O(n^4) method
S, V = torch.linalg.eigh(H)
S_sqrt = torch.sqrt(torch.clamp(S, min=0))
H_sqrt = V @ torch.diag(S_sqrt) @ V.T
tr_H_sqrt = torch.diagonal(H_sqrt).sum()
W_flat = W.reshape(1, 2**k, 2**k)
W_reshaped = hb_transform_reshape(W_flat, k).reshape(1, n*n)
coeffs = hb_transform_loop(W_reshaped, k) / (2**k)
coeffs = coeffs.reshape(n*n)
norm = 1/n
coeffs = coeffs / norm
Wscale = coeffs.square().mean().sqrt() / cb.opt_scale
coeffs = coeffs / Wscale
base_cliques = torch.tensor(cliques, dtype=torch.long)
num_cliques = n*n // 64
offsets = torch.arange(num_cliques, dtype=torch.long).view(-1, 1, 1) * 64
cliques_8 = offsets + base_cliques.unsqueeze(0)
cliques_8 = cliques_8.reshape(-1, 8)
W_flat = W.reshape(1, 2**k, 2**k)
W_reshaped = hb_transform_reshape(W_flat, k).reshape(1, n*n)
coeffs = hb_transform_loop(W_reshaped, k) / (2**k)
coeffs = coeffs.reshape(1, n*n)
norm = 1/n
coeffs = coeffs / norm
Wscale = coeffs.square().mean().sqrt() / cb.opt_scale
coeffs = coeffs / Wscale
hat_coeffs = torch.zeros_like(coeffs)
row_i = torch.arange(n)
buffer = 256
perms = torch.zeros(n * n, n, dtype=torch.long)
signs = torch.zeros(n * n, n)
for start in range(0, n*n, buffer):
    end = min(start + buffer, n*n)
    identity = torch.zeros(end - start, n*n)
    identity[torch.arange(end - start), torch.arange(start, end)] = 1.0
    B = hb_transform(identity) * (2**k)
    perms[start:end] = B.abs().argmax(dim=2).long()
    signs[start:end] = B[torch.arange(end-start)[:, None], row_i[None, :], perms[start:end]]
del B, identity
hat_coeffs = torch.zeros_like(coeffs)
for ci in range(len(cliques_8)):
    clique = cliques_8[ci]
    target = coeffs[:, clique].clone()
    if ci > 0:
        correction = torch.zeros(8, dtype=coeffs.dtype)
        correction = torch.zeros(8, dtype=coeffs.dtype)
        p_i = perms[clique]
        s_i = signs[clique]
        for cp in range(ci):
            prior_clique = cliques_8[cp]
            prev_err = (coeffs[:, prior_clique] - hat_coeffs[:, prior_clique]).squeeze(0)
            p_j = perms[prior_clique]
            s_j = signs[prior_clique]                
            p_j_t = torch.zeros_like(p_j)
            p_j_t.scatter_(1, p_j, row_i.expand(8, -1))                
            s_j_t = torch.zeros_like(s_j)
            s_j_t.scatter_(1, p_j, s_j)
            p_i_exp = p_i.unsqueeze(1).expand(-1, 8, -1) # (8, 8, n)
            s_i_exp = s_i.unsqueeze(1).expand(-1, 8, -1)
            p_j_t_exp = p_j_t.unsqueeze(0).expand(8, -1, -1)
            s_j_t_exp = s_j_t.unsqueeze(0).expand(8, -1, -1)
            pi_times_pj_t = torch.gather(p_j_t_exp, 2, p_i_exp)
            si_times_sj_t = s_i * torch.gather(s_j_t_exp, 2, p_i_exp)
            H_curr = H_sqrt[row_i.view(1, 1, -1), pi_times_pj_t] # (8, 8, n)
            cross = (H_curr * si_times_sj_t).sum(dim=2) / tr_H_sqrt / (2**k)**2 # (8, 8)
            correction += cross @ prev_err
        target += correction.unsqueeze(0)
    hat_clique, _ = cb.quantize(target)
    hat_coeffs[:, clique] = hat_clique
hatW_naive = hb_transform(hat_coeffs).reshape(n, n) * Wscale
error = (W - hatW_naive).norm() / W.norm()
print(f"Error: {error}")

Error: 0.30011532885937714


In [27]:
torch.mean((hatW_naive - hatW_fast) ** 2)

tensor(0.0005)

In [28]:
n = 64
k = calculate_k(n * n)
torch.manual_seed(0)
W = torch.randn(n, n)
A = torch.randn(n, n)
H = (A + A.T) / 2
H = H + torch.eye(n) * (abs(torch.linalg.eigvalsh(H).min()) + 1.0)
hatW_fast = fast_orth_quant_full(W, H, n_full=n, n_sub=64)
err = (W - hatW_fast).norm() / W.norm()
print(f"Error: {err.item():.6f}")
proxy_loss = ((hatW_fast - W) @ H @ (hatW_fast - W).T).trace()
print(f"Proxy loss: {proxy_loss.item():.6e}")
S, V = torch.linalg.eigh(H)
S_sqrt = torch.sqrt(torch.clamp(S, min=0))
H_sqrt = V @ torch.diag(S_sqrt) @ V.T
tr_H_sqrt = torch.diagonal(H_sqrt).sum()
W_flat = W.reshape(1, 2**k, 2**k)
W_reshaped = hb_transform_reshape(W_flat, k).reshape(1, n*n)
coeffs = hb_transform_loop(W_reshaped, k) / (2**k)
coeffs = coeffs.reshape(n*n)
norm = 1/n
coeffs = coeffs / norm
Wscale = coeffs.square().mean().sqrt() / cb.opt_scale
coeffs = coeffs / Wscale
base_cliques = torch.tensor(cliques, dtype=torch.long)
num_cliques = n*n // 64
offsets = torch.arange(num_cliques, dtype=torch.long).view(-1, 1, 1) * 64
cliques_n = offsets + base_cliques.unsqueeze(0)
cliques_n = cliques_n.reshape(-1, 8)

def recursive_clique_order(n, base_cliques):
    if n == 8:
        return list(range(8))
    cliques_per_group = (n * n) // 32
    order = []
    for b in range(4):
        next_order = recursive_clique_order(n//2, base_cliques)
        offset = b * cliques_per_group
        order.extend([offset + i for i in next_order])
    return order
order = recursive_clique_order(n, cliques)

J = torch.zeros((n*n, n*n), dtype=H_sqrt.dtype)
basis_n = hb_transform(torch.eye(n*n)) * (2**k)
for i in range(n*n):
    J[i] = ((basis_n[i] @ H_sqrt).unsqueeze(0) * basis_n).sum(dim=(1, 2)) / tr_H_sqrt / (n ** 2)

hat_coeffs = torch.zeros_like(coeffs)
error = torch.zeros(n * n, dtype=coeffs.dtype)
for i, ci in enumerate(order):
    target = coeffs[cliques_n[ci]].clone()
    if i > 0:
        correction = torch.zeros(8, dtype=coeffs.dtype)
        for cp in range(i):
            prior_clique = order[cp]
            correction += J[cliques_n[ci]][:, cliques_n[prior_clique]] @ error[cliques_n[cp]]
        target += correction
    hat_clique, _ = cb.quantize(target.unsqueeze(0))
    hat_coeffs[cliques_n[ci]] = hat_clique.squeeze(0)
    error[cliques_n[ci]] = coeffs[cliques_n[ci]] - hat_coeffs[cliques_n[ci]]
hatW_J = hb_transform(hat_coeffs.reshape(1, n*n)).reshape(n, n) * Wscale
err = (W - hatW_J).norm() / W.norm()
print(f"Error (O(n^4), using J): {err}")

row_i = torch.arange(n)
buffer = 256
perms = torch.zeros(n * n, n, dtype=torch.long)
signs = torch.zeros(n * n, n)
for start in range(0, n*n, buffer):
    end = min(start + buffer, n*n)
    identity = torch.zeros(end - start, n*n)
    identity[torch.arange(end - start), torch.arange(start, end)] = 1.0
    B = hb_transform(identity) * (2**k)
    perms[start:end] = B.abs().argmax(dim=2).long()
    signs[start:end] = B[torch.arange(end-start)[:, None], row_i[None, :], perms[start:end]]
del B, identity
hat_coeffs = torch.zeros_like(coeffs)
for i, ci in enumerate(order):
    clique = cliques_8[ci]
    target = coeffs[clique].clone()
    if i > 0:
        correction = torch.zeros(8, dtype=coeffs.dtype)
        correction = torch.zeros(8, dtype=coeffs.dtype)
        p_i = perms[clique]
        s_i = signs[clique]
        for cp in range(i):
            prior_clique = cliques_8[cp]
            prev_err = (coeffs[prior_clique] - hat_coeffs[prior_clique]).squeeze(0)
            p_j = perms[prior_clique]
            s_j = signs[prior_clique]                
            p_j_t = torch.zeros_like(p_j)
            p_j_t.scatter_(1, p_j, row_i.expand(8, -1))                
            s_j_t = torch.zeros_like(s_j)
            s_j_t.scatter_(1, p_j, s_j)
            p_i_exp = p_i.unsqueeze(1).expand(-1, 8, -1) # (8, 8, n)
            s_i_exp = s_i.unsqueeze(1).expand(-1, 8, -1)
            p_j_t_exp = p_j_t.unsqueeze(0).expand(8, -1, -1)
            s_j_t_exp = s_j_t.unsqueeze(0).expand(8, -1, -1)
            pi_times_pj_t = torch.gather(p_j_t_exp, 2, p_i_exp)
            si_times_sj_t = s_i * torch.gather(s_j_t_exp, 2, p_i_exp)
            H_curr = H_sqrt[row_i.view(1, 1, -1), pi_times_pj_t] # (8, 8, n)
            cross = (H_curr * si_times_sj_t).sum(dim=2) / tr_H_sqrt / (2**k)**2 # (8, 8)
            correction += cross @ prev_err
        target += correction
    hat_clique, _ = cb.quantize(target.unsqueeze(0))
    hat_coeffs[clique] = hat_clique.squeeze(0)
hatW_order = hb_transform(hat_coeffs).reshape(n, n) * Wscale
err = (W - hatW_order).norm() / W.norm()
print(f"Error (O(n^4), using naive and order): {err}")

print(f"MSE of n^3 and using J: {((hatW_fast - hatW_J)**2).mean()}")
print(f"MSE of n^3 and using naive with recursive order: {((hatW_fast - hatW_order)**2).mean()}")

Error: 0.300115
Proxy loss: 4.297903e+03
Error (O(n^4), using J): 0.30011520417545234
Error (O(n^4), using naive and order): 0.30011532885937714
MSE of n^3 and using J: 1.5061444547876855e-32
MSE of n^3 and using naive with recursive order: 0.00045696329952534843
